# Guide to model training and inference

This shows the basics of implementing models for video summarization, while also noting 

In [1]:
from Data import SingleH5Loader,MultiH5Loader
from Models import PGL_SUM
import json
import os
import pytorch_lightning as pl
import torch.nn as nn
from torch.utils.data import Dataset
import torch.nn.functional as F

# Initial Steps

We hold the splits in jsons
The splits have the following structure

Train:

Dataset_name/key_name


Val:

Dataset_name/key_name

Test:

Dataset_name/key_name

In [2]:
split_file = 'Splits/summe_can_1.json'
split_name = 'train'
cross_val_idx = 0
feature_name = 'googlenet'
included_dataset = ['summe']

In [ ]:
import h5py
base_file_path = 'Data/h5Datasets'

In [9]:

class MultiH5Loader (Dataset):
  def __init__(self,split_file,split_name,cross_val_idx,feature_name,included_dataset):
    with open(split_file,'r') as f:
      self.split_file = json.load(f)
    self.data_points = self.split_file[cross_val_idx][split_name]
    self.feature_name = feature_name
    self._create_data_dict(included_dataset)

  def __len__(self):
    return len(self.data_points)

  def _create_data_dict(self,included_datasets):
    self.dataset_dict = {}
    for dataset in included_datasets:
      data_paths = os.path.join(base_file_path,f'{feature_name}',f'{self.feature_name}_{dataset}.h5')
      self.dataset_dict[dataset]= h5py.File(data_paths,'r')


  def __getitem__(self,idx):
    data_point = self.data_points[idx]
    dataset,video_index = data_point.split('/')
    features = self.dataset_dict[dataset][video_index]['features'][...]
    gtscore = self.dataset_dict[dataset][video_index]['gtscore'][...]
    return features,gtscore,data_point

In [10]:
a = MultiH5Loader(split_file,split_name,cross_val_idx,feature_name,included_dataset)

In [11]:
next(iter(a))

(array([[0.00000000e+00, 0.00000000e+00, 3.61840939e-04, ...,
         1.02224345e-04, 2.28259116e-02, 0.00000000e+00],
        [0.00000000e+00, 0.00000000e+00, 3.40301369e-04, ...,
         3.51528870e-03, 4.94062938e-02, 0.00000000e+00],
        [0.00000000e+00, 5.40758716e-04, 2.82520137e-04, ...,
         4.78237853e-05, 2.36145183e-02, 0.00000000e+00],
        ...,
        [0.00000000e+00, 0.00000000e+00, 1.80268151e-04, ...,
         3.99936456e-03, 2.72235880e-03, 0.00000000e+00],
        [0.00000000e+00, 0.00000000e+00, 1.11069101e-04, ...,
         1.36979891e-03, 1.36739679e-03, 0.00000000e+00],
        [0.00000000e+00, 0.00000000e+00, 1.44374644e-04, ...,
         2.81268102e-03, 8.94312747e-03, 0.00000000e+00]], dtype=float32),
 array([0.        , 0.        , 0.        , 0.        , 0.06666667,
        0.06666667, 0.06666667, 0.06666667, 0.13333334, 0.13333334,
        0.13333334, 0.13333334, 0.06666667, 0.06666667, 0.13333334,
        0.06666667, 0.13333334, 0.13333334, 0.

In [ ]:
train_data = SingleH5Loader(split_file,'train',0,'googlenet','summe')


In [ ]:
class MetadataStore:
    def __init__(self, datasets,dataset_paths_dict:dict = None):
        # The dataset dict paths should also allow you to override and add custom h5's incase the h5's deviate (different shot boundaries,fps sampling etc)

        self.datasets = datasets
        self.dataset_path_dicts = dataset_paths_dict
        self.files = {}

    def open(self):
        if self.dataset_path_dicts:
            self.files = {dataset:h5py.File(paths) for dataset,paths in self.dataset_path_dicts.items()}
        else:
            self.files = {
                dataset: h5py.File(
                    f"Data/Metadata/{dataset}_metadata.h5", "r"
                )
                for dataset in self.datasets
            }

    def get(self, dataset, video_key):
        f = self.files[dataset]
        group = f[video_key]

        metadata = {
            "positions": group["positions"][...],
            "n_frames": int(group["n_frames"][...]),
            "shot_bounds": group["shot_bounds"][...]
        }


        return metadata

    def close(self):
        for f in self.files.values():
            f.close()
        self.files.clear()

In [15]:
from collections.abc import Callable

In [ ]:
# Training of basic models which only require a single loss function to optimize
#TODO: Update metadata stores to also return the GT features we want to use
#TODO
class BaseTrainer(pl.LightningModule):

    def __init__(self,model:nn.Module,datasets:list|dict,eval_type:dict[str],criterion:Callable = F.mse_loss,eval_criterion='corr'):
        self.model = model
        
        self.metadata_stores = []
        self.val_preds = []
        self.gts = []
        self.eval_criterion = self.eval_criterion
        #TODO, add validation keys to exclude for hyper-parameter logging
        self.criterion = criterion

    def training_step(self, batch, batch_idx):
        x, y,_ = batch
        
        # Forward pass
        y_pred = self.model(x)
        
        # Calculate loss (MSE)
        loss = self.criterion(y_pred, y)
        
        # Log the loss
        self.log('train_loss', loss)
        return loss

    def validation_step(self,batch,batch_idx):
        x,y,video_key = batch # This might need to be changed to a dict, check with collate function
        y_pred = self.model(x).to('cpu') #TODO: maybe change this to a dictionary output.
        y = y.to('cpu')
        dataset,video_index = video_key.split('/')
        metadata = self.metadata_stores.get(dataset,video_index)
        ground_truth_data = self.metadata_stores.get_gt(dataset,video_index)
        eval_type()
        if self.eval_criterion =='corr':
            kendall,spearman = process_and_route_single(y_pred,y,metadata,ground_truth_data)



        